In [1]:
# testing the kernel for scmpra package
import scMPRAforge as scm 

2025-11-10 10:01:25.160241: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-10 10:01:26.063174: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.95.3/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [11]:

import pandas as pd
import urllib.request
import subprocess 
import h5py
from scipy.sparse import csc_matrix
import scanpy as sc

ModuleNotFoundError: No module named 'scanpy'

In [4]:
data_root="/home/sxl6/palmer_scratch/tabula_data"
name="GSE269037_RAW"
seelig_data_path=f"{data_root}/{name}.tar"
#paper is here https://www.cell.com/cell-systems/fulltext/S2405-4712%2825%2900135-8
# the GEO dataset https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE269037
urllib.request.urlretrieve("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE269037&format=file",
                           seelig_data_path)

('/home/sxl6/palmer_scratch/tabula_data/GSE269037_RAW.tar',
 <http.client.HTTPMessage at 0x145fcf4572e0>)

In [4]:
subprocess.run(['tar', '-xf', seelig_data_path, '-C', f"{data_root}"], check=True)

CompletedProcess(args=['tar', '-xf', '/home/sxl6/palmer_scratch/tabula_data/GSE269037_RAW.tar', '-C', '/home/sxl6/palmer_scratch/tabula_data'], returncode=0)

extrated two files 
GSM8305416_R1-scMPRA.h5  GSM8305417_R1-scMPRA_tx.h5

In [5]:
file_1=h5py.File(f"{data_root}/GSM8305416_R1-scMPRA.h5",'r')
file_2=h5py.File(f"{data_root}/GSM8305417_R1-scMPRA_tx.h5")
group = list(file_1.keys())[0]
#print(group)
# repeat for file 2 
group = list(file_2.keys())[0]
print(group)


unknown


note: my file2 is the tx file 

In [6]:
print(list(file_1[list(file_1.keys())[0]]))
print(list(file_2[list(file_2.keys())[0]]))


['barcodes', 'data', 'gene_names', 'genes', 'indices', 'indptr', 'shape']
['barcodes', 'data', 'gene_names', 'genes', 'indices', 'indptr', 'shape']


In [7]:
def decode(handle):
    unknown_group = handle['unknown']
    
    # Load the sparse matrix components
    data = unknown_group['data'][:]
    indices = unknown_group['indices'][:]
    indptr = unknown_group['indptr'][:]
    shape = unknown_group['shape'][:]
    
    # Load barcodes and gene names
    barcodes = unknown_group['barcodes'][:].astype(str)
    genes = unknown_group['genes'][:].astype(str)
    genes = [gene.decode('utf-8') for gene in unknown_group['genes'][:]]
    gene_names = unknown_group['gene_names'][:].astype(str)

    sparse_matrix = csc_matrix((data, indices, indptr), shape=shape)

    # Determine which has fewer unique values for the outer index
    multi_index=None
    if len(set(genes)) < len(set(gene_names)):
        multi_index = pd.MultiIndex.from_tuples(list(zip(genes, gene_names)), names=['Gene', 'Gene Name'])
    else:
        multi_index = pd.MultiIndex.from_tuples(list(zip(gene_names, genes)), names=['Gene Name', 'Gene'])

    df= pd.DataFrame.sparse.from_spmatrix(sparse_matrix, index=multi_index, columns=barcodes)
    df.columns.name = 'Cell Barcode'
    return df

    
MPRA=decode(file_1)
GEX=decode(file_2)

In [ ]:
flatty=MPRA.reset_index()
# gene name and gene are identical 
all(flatty["Gene Name"]==flatty["Gene"])

True

In [ ]:
#save to csv file so i don't have to run previous cells again 
flatty.to_csv(f"{data_root}/seelig_mpra_unpro.tsv",sep="\t")

In [10]:
data=pd.read_csv(f"{data_root}/seelig_mpra_unpro.tsv",index_col=0,sep="\t")
# set the index of the rows to be gene 
data.index=data['Gene']
# delete the gene name and gene columns 
data.drop(['Gene Name','Gene'],axis=1,inplace=True)
data=data.stack().reset_index()
data.rename({'level_1': 'cell_bc',0:'umis_mpra_bc'},axis=1,inplace=True)
print(data.head())
# turn umi counts into integers
data['umis_mpra_bc']=data['umis_mpra_bc'].astype(int)


                                                Gene   cell_bc  umis_mpra_bc
0  AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...  A9_A2_A2           0.0
1  AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...  A6_A2_A2           0.0
2  AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...  A2_B1_A2           0.0
3  AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...  A5_B2_A2           0.0
4  AAAATATCTCTGTAGGCAGATGCTTACAGCTGCTGCCGCAGACATA...  A1_B2_A2           0.0


In [ ]:
sc.pl.umap(data, color="umis_mpra_bc")